# Python II — Object-Oriented Programming
### Lesson 1 — Solutions to the exercises

There is often more than one correct solution: the ones here aim to be **simple and readable** rather than clever.

| # | Exercise | Concepts used |
|---|----------|---------------|
| 1 | `Bird` class | inheritance, overriding, `super()` |
| 2 | `Shelter.adopt()` | methods that modify state, raising exceptions |
| 3 | `count_calls` decorator | decorators, functions as objects |
| 4 | `__iter__` | special methods, iteration protocol |

At the end you find all the solutions **merged into the complete example**.

In [12]:
# Code of previous notebook

def log_call(func):      # we slim down the decorator to log only the function name and arguments and not include @functools.wraps(func) even if not good practice!
    def wrapper(*args, **kwargs):
        print(f"[LOG] {func.__name__}{args[1:]}")      # skip self in the log
        return func(*args, **kwargs)
    return wrapper


class Animal:
    kingdom = "Animalia"                 # class attribute
    count = 0

    def __init__(self, name, age):
        self.name = name                 # instance attributes
        self.age = age
        Animal.count += 1

    def speak(self):
        # Generic animals do not know what to say:
        # every subclass is expected to override this method.
        #raise NotImplementedError(f"{type(self).__name__} must implement speak()")
        ...

    def introduce(self):
        print(f"I am {self}, and I say: {self.speak()}")

    @log_call
    def birthday(self):
        self.age += 1

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

    def __str__(self):
        return f"{self.name} ({self.age} y.o.)"

    def __eq__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return (type(self), self.name, self.age) == (type(other), other.name, other.age)

    def __lt__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return self.age < other.age


class Mammal(Animal):
    is_hairy = True


class Dog(Mammal):
    def __init__(self, name, age, breed="unknown"):
        super().__init__(name, age)
        self.breed = breed

    def speak(self):
        return "Woof!"

    def introduce(self):                 # overriding + super()
        super().introduce()
        print(f"   ...and I am a {self.breed}.")


class Cat(Mammal):
    def speak(self):
        return "Meow!"


class Shelter:
    def __init__(self, name, animals=None):
        self.name = name
        self.animals = []
        for animal in (animals if animals is not None else []):
            self.admit(animal)               # reuse the check

    def admit(self, animal):
        if not isinstance(animal, Animal):
            raise TypeError(f"Shelter only accepts Animal instances, got {type(animal).__name__}")
        self.animals.append(animal)

    def __repr__(self):
            return f"{type(self).__name__}({self.name!r}, {self.animals!r})"
    
    def __len__(self):
        return len(self.animals)

    def __getitem__(self, index):
        return self.animals[index]

    def __contains__(self, animal):
        return animal in self.animals

    def __add__(self, other):
        if not isinstance(other, Shelter):
            return NotImplemented
        return Shelter(f"{self.name} & {other.name}", self.animals + other.animals)

In [14]:
north = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat("Luna", 5)])   # good Shelter instance
print(north)

try:
    northWrong = Shelter("NorthWrong", ["Rex", "Luna"])   # wrong Shelter instance, will raise TypeError
except TypeError as e:
    print("TypeError:", e)

Shelter('North', [Dog('Rex', 3), Cat('Luna', 5)])
TypeError: Shelter only accepts Animal instances, got str


In [15]:
## OLD VERSION: In principle, since no checks are implemented, the Shelter class could accept any iterable object, say a list of everything, including strings and numbers, not just instances of Animal.
#               This will not raise error when instanciating the class, but it will when we try to access instance attributes of the elements in the list, which are not Animal instances.
class ShelterOld:
    def __init__(self, name, animals=None):
        self.name = name
        self.animals = list(animals) if animals is not None else []

    def admit(self, animal):
        self.animals.append(animal)

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.animals!r})"

    def __len__(self):
        return len(self.animals)

    def __getitem__(self, index):
        return self.animals[index]

    def __contains__(self, animal):
        return animal in self.animals

    def __add__(self, other):
        if not isinstance(other, Shelter):
            return NotImplemented
        return Shelter(f"{self.name} & {other.name}", self.animals + other.animals)

In [16]:
northWrong = ShelterOld("NorthWrong", [1, "Luna"])   # this instanciate the class
northWrong.admit("Stella")
print(northWrong)

ShelterOld('NorthWrong', [1, 'Luna', 'Stella'])


---
## Exercise 1 — A `Bird` class

> Add a `Bird(Animal)` class that implements `speak()` and has an extra attribute `can_fly` (default `True`).

**Reasoning**

- `Bird` inherits from `Animal` directly (a bird is not a mammal).
- `Animal.speak()` is empty, so `Bird` must **override** `speak()` with its own version,
  otherwise `introduce()` would fail.
- To add `can_fly` we **override `__init__`**. We let the parent set `name` and `age` with `super().__init__(name, age)`.
- Giving `can_fly=True` as a **default value** means we only need to pass it for birds that cannot fly.
- As a bonus we also override `introduce()` (extending it with `super()`), and `__repr__` to show the new attribute.

In [17]:
class Bird(Animal):
    """A class representing a bird, which is a type of animal."""      # good practice to include a docstring for classes, which describes the purpose of the class

    def __init__(self, name, age, can_fly=True):
        super().__init__(name, age)      # inherit from Animal name, age
        self.can_fly = can_fly           # define the new instance attribute

    def speak(self):                     # overrides Animal.speak
        return "Tweet!"

    def introduce(self):
        super().introduce()              # inherit the introduction from Animal + add the flying ability
        if self.can_fly:                 
            print("   ...and I can fly!")
        else:
            print("   ...but I cannot fly.")

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r}, can_fly={self.can_fly!r})"

**Test.** Since `can_fly` has a default value, we only need to pass it for birds that cannot fly.

In [18]:
tweety = Bird("Tweety", 2)
pingu = Bird("Pingu", 4, can_fly=False)
polly = Bird("Polly", 10)

for bird in (tweety, pingu, polly):
    bird.introduce()

print()      # just print an empty line to separate the output of the previous cell from the next one
print(repr(pingu))

print()
print(isinstance(pingu, Animal), isinstance(pingu, Mammal))   # expected output: True False

I am Tweety (2 y.o.), and I say: Tweet!
   ...and I can fly!
I am Pingu (4 y.o.), and I say: Tweet!
   ...but I cannot fly.
I am Polly (10 y.o.), and I say: Tweet!
   ...and I can fly!

Bird('Pingu', 4, can_fly=False)

True False


---
## Exercise 2 — `Shelter.adopt()`

> Give `Shelter` an `adopt(name)` method that removes and returns the animal with that name.

**Reasoning**

- We look through the list of animals, keeping track of the **position** of each one with `enumerate`.
- As soon as we find an animal with the right name, `list.pop(index)` **removes it and returns it** in a single step,
  and we immediately `return` it (so if two animals have the same name, only the first one is adopted).
- **OPTIONAL**: If the loop ends without finding anything, we reach the last line and **raise** a `KeyError` with a clear message.
  Raising an exception is better than returning `None`: the caller cannot silently ignore the problem.


We re-define the whole `Shelter` class, adding the new method.

In [31]:
class Shelter:
    def __init__(self, name, animals=None):
        self.name = name
        self.animals = []
        for animal in (animals if animals is not None else []):
            self.admit(animal)               
            
    def admit(self, animal):
        if not isinstance(animal, Animal):
            raise TypeError(f"Shelter only accepts Animal instances, got {type(animal).__name__}")
        self.animals.append(animal)

    # NEW METHOD
    def adopt(self, name):
        for index, animal in enumerate(self.animals):  # enumerate returns a tuple of (index, animal) for each animal in the list, so we can acess both the index and the animal object in the same loop
            if animal.name == name:
                return self.animals.pop(index)         # return the animal object and remove it from the shelter's list of animals
        raise KeyError(f"No animal named {name!r} in shelter {self.name!r}")      # Even if the animal is not in Shelter and no error is raise, it will still work and return None.

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.animals!r})"

    def __len__(self):
        return len(self.animals)

    def __getitem__(self, index):
        return self.animals[index]

    def __contains__(self, animal):
        return animal in self.animals

    def __add__(self, other):
        if not isinstance(other, Shelter):
            return NotImplemented
        return Shelter(f"{self.name} & {other.name}", self.animals + other.animals)

In [32]:
shelter = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat("Luna", 5), Bird("Tweety", 2)])
print("Before:", shelter, " Total animals:", len(shelter))

my_pet = shelter.adopt("Luna")
print("Adopted:", my_pet)
print("After: ", shelter, " Total animals:", len(shelter))


try:
    shelter.adopt("Garfield")
except KeyError as err:
    print("KeyError:", err)

Before: Shelter('North', [Dog('Rex', 3), Cat('Luna', 5), Bird('Tweety', 2, can_fly=True)])  Total animals: 3
Adopted: Luna (5 y.o.)
After:  Shelter('North', [Dog('Rex', 3), Bird('Tweety', 2, can_fly=True)])  Total animals: 2
KeyError: "No animal named 'Garfield' in shelter 'North'"


---
## Exercise 3 — A `count_calls` decorator

> Write a decorator `count_calls` that counts how many times a function has been called
> (hint: you can store the counter as an attribute of `wrapper`).

**Reasoning**

We use the same structure as the `log_call` decorator seen in the lesson:
- `count_calls(func)` defines and returns a `wrapper` that calls `func`;
- the counter must **survive between calls**, so it cannot be a local variable of `wrapper` (it would be reset
  every time). Since functions are objects, we can attach an attribute to `wrapper` itself: `wrapper.calls = 0`;
- each time `wrapper` runs it does `wrapper.calls += 1`, then calls the original function and returns its result.

Since the decorated name *is* `wrapper` (remember: `@count_calls` means `f = count_calls(f)`),
the counter can be read from outside as `f.calls`.

In [34]:
import functools

def count_calls(func):
    @functools.wraps(func)     # again good practice, but not compulsory
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0        # initialised once, when the function is decorated
    return wrapper

In [35]:
@count_calls
def greet(name):
    return f"Hello {name}!"

print(greet.calls)
greet("Rex")
greet("Luna")
print(greet("Tom"))
print("greet was called", greet.calls, "times")

0
Hello Tom!
greet was called 3 times


In [ ]:
### WRONG SOLUTION: initialising the counter inside the wrapper function would reset it to 0 every time the function is called
def count_calls_wrong(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls = 0        # WRONG: runs on every call
        wrapper.calls += 1
        return func(*args, **kwargs)
    return wrapper

@count_calls_wrong    # decorate greet with wrong initialisation of the counter
def greet(name):
    return f"Hello {name}!"

greet("Rex")
greet("Luna")
print(greet("Tom"))
print("greet was called", greet.calls, "times")   # expected output? 

**On a method.** The decorator works on methods as well. Notice, however, that the counter is attached to the
function, and the function belongs to the **class**: it counts the calls made by **all instances together**.

In [ ]:
class Parrot(Bird):
    @count_calls
    def speak(self):
        return "Hello!"

polly = Parrot("Polly", 10)
kiwi = Parrot("Kiwi", 3)

polly.introduce()    # introduce() calls speak() which is decorated with count_calls, so the counter will be incremented
kiwi.introduce()
kiwi.speak()

print("Parrot.speak was called", Parrot.speak.calls, "times (by all parrots together)")

---
## Exercise 4 — `__iter__`

> Make `Shelter` iterable in a more explicit way by implementing `__iter__`.
> Does the `for` loop still work if you remove `__getitem__`? And indexing?

**Reasoning**

When Python runs `for x in obj`, it first calls `iter(obj)`, which uses `obj.__iter__()` and expects an
**iterator**: an object that gives out the elements one at a time.
In the lesson our loop worked thanks to an old *fallback*: if `__iter__` is missing, Python tries
`__getitem__(0)`, `__getitem__(1)`, ... until an `IndexError`.

Our animals are already stored in a list, and a list knows how to produce an iterator, so the simplest
implementation just **delegates** to it: `return iter(self.animals)`.

To answer the question we write a version of `Shelter` with `__iter__` and **without** `__getitem__`.

In [ ]:
class IterShelter:
    def __init__(self, name, animals=None):
        self.name = name
        self.animals = list(animals) if animals is not None else []

    def __repr__(self):
        return f"IterShelter({self.name!r}, {self.animals!r})"

    def __len__(self):
        return len(self.animals)

    def __iter__(self):                 # new dunder method!
        return iter(self.animals)       # delegate to the list's iterator

    # no __getitem__ on purpose
    # no __contains__ on purpose

**Iteration still works**, and so does everything built on top of it: `list()`, `sorted()`, `max()`, unpacking, ...

In [ ]:
south = IterShelter("South", [Dog("Fido", 8, "Beagle"), Cat("Tom", 1), Bird("Tweety", 2)])

for animal in south:
    print(" -", animal)

print(sorted(south))
print("Oldest:", max(south))

**Indexing does not work**: `__iter__` only lets us walk through the elements in order, it does not give
random access. Square brackets require `__getitem__`.

In [ ]:
try:
    south[0]
except TypeError as err:
    print("TypeError:", err)

**And `in`?** We did not define `__contains__` either, yet `in` works: when `__contains__` is missing,
Python falls back to iterating and comparing each element with `==` (our `__eq__`).

In [ ]:
print(Cat("Tom", 1) in south)
print(Cat("Tom", 2) in south)

---
## All the solutions together

Here is the complete example with every solution merged in:
- the new `Bird` class (exercise 1);
- `Dog.speak` is decorated with `count_calls` (exercise 3);
- `Shelter` has `adopt` (exercise 2) and an explicit `__iter__` (exercise 4). We keep `__getitem__` as well, so that indexing still works.

In [36]:
def count_calls(func):                                     # exercise 4
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper


class Animal:
    kingdom = "Animalia"
    count = 0

    def __init__(self, name, age):
        self.name = name
        self.age = age
        Animal.count += 1

    def speak(self):
        # Generic animals do not know what to say:
        # every subclass is expected to replace (override) this method.
        raise NotImplementedError(f"{type(self).__name__} must implement speak()")

    def introduce(self):
        print(f"I am {self}, and I say: {self.speak()}")

    def birthday(self):
        self.age += 1
        print(f"Happy birthday {self.name}! Now {self.age} years old.")

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

    def __str__(self):
        return f"{self.name} ({self.age} y.o.)"

    def __eq__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return (type(self), self.name, self.age) == (type(other), other.name, other.age)

    def __lt__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return self.age < other.age


class Mammal(Animal):
    is_hairy = True


class Dog(Mammal):
    def __init__(self, name, age, breed="unknown"):
        super().__init__(name, age)
        self.breed = breed
    
    @count_calls                        # exercise 4: it only counts the time speaks is invoked for Dog instances, not for other animals.
    def speak(self):
        return "Woof!"

    def introduce(self):
        super().introduce()
        print(f"   ...and I am a {self.breed}.")


class Cat(Mammal):
    def speak(self):
        return "Meow!"


class Bird(Animal):                                        # exercise 1
    def __init__(self, name, age, can_fly=True):
        super().__init__(name, age)
        self.can_fly = can_fly

    def speak(self):           
        return "Tweet!"

    def introduce(self):
        super().introduce()
        print("   ...and I can fly!" if self.can_fly else "   ...but I cannot fly.")

    def __repr__(self):
        return f"Bird({self.name!r}, {self.age!r}, can_fly={self.can_fly!r})"


class Shelter:
    def __init__(self, name, animals=None):
        self.name = name
        self.animals = []
        for animal in (animals if animals is not None else []):
            self.admit(animal)               
            
    def admit(self, animal):
        if not isinstance(animal, Animal):
            raise TypeError(f"Shelter only accepts Animal instances, got {type(animal).__name__}")
        self.animals.append(animal)

    def adopt(self, name):                                 # exercise 2
        for index, animal in enumerate(self.animals):
            if animal.name == name:
                return self.animals.pop(index)
        raise KeyError(f"No animal named {name!r} in shelter {self.name!r}")

    def __repr__(self):
        return f"Shelter({self.name!r}, {self.animals!r})"

    def __len__(self):
        return len(self.animals)

    def __iter__(self):                                    # exercise 5
        return iter(self.animals)

    def __getitem__(self, index):
        return self.animals[index]

    def __contains__(self, animal):
        return animal in self.animals

    def __add__(self, other):
        if not isinstance(other, Shelter):
            return NotImplemented
        return Shelter(f"{self.name} & {other.name}", self.animals + other.animals)

In [37]:
north = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat("Luna", 5)])
south = Shelter("South", [Bird("Pingu", 4, can_fly=False), Dog("Fido", 8, "Beagle")])
everyone = north + south

for animal in sorted(everyone):                  # __iter__ + __lt__
    animal.introduce()

print("\nDog.speak called", Dog.speak.calls, "times")
print(Dog.speak.__name__)      # if no @functools.wraps(func) in the decorator, this would print "wrapper" instead of "speak"

adopted = everyone.adopt("Pingu")                 # adopt
print("Adopted:", adopted, "| animals left:", len(everyone))    # BUT is Pingu still in south?


I am Rex (3 y.o.), and I say: Woof!
   ...and I am a Labrador.
I am Pingu (4 y.o.), and I say: Tweet!
   ...but I cannot fly.
I am Luna (5 y.o.), and I say: Meow!
I am Fido (8 y.o.), and I say: Woof!
   ...and I am a Beagle.

Dog.speak called 2 times
speak
Adopted: Pingu (4 y.o.) | animals left: 3
